# Human RefSeq transcript coding-status ML study

**Question:** Can simple sequence features distinguish human RefSeq transcripts labeled by accession prefix as protein-coding (`NM_`) versus non-coding (`NR_`)? FLIP2 and variant information are intentionally out of scope.

This notebook verifies the NCBI assembly and annotation release, audits accession-prefix labels against direct transcript-level GFF keys, and reports parent-gene biotype disagreements separately. It samples up to 3,000 records per label, clusters **every sampled transcript** with MMseqs2 in explicit nucleotide database mode at **80% identity and 80% coverage**, then makes connected components joining shared GeneIDs and sequence clusters indivisible across train/validation/test splits. It retains short non-coding RNAs and reports errors by biotype.


## 0. Assumptions and minimum-length rule

The operational label is `NM_ = 1` and `NR_ = 0`; it is a RefSeq convention, not an independent functional assay. The only length rule is **`length >= 20 nt`**, used as a malformed/empty-sequence QC floor, not a biological filter. It is deliberately low enough to retain miRNA, tRNA, snRNA, and other legitimate short RNAs; length is reported because it can otherwise become a shortcut.

In [ ]:
# Reproducible source, grouping, and ML configuration.
import os, re, sys, json, gzip, shutil, zipfile, subprocess, random, hashlib, platform
from pathlib import Path
from datetime import datetime, timezone
from collections import Counter
from urllib.parse import unquote
import numpy as np, pandas as pd
SEED=42; ACCESSION="GCF_000001405.40"; EXPECTED_ASSEMBLY="GRCh38.p14"
EXPECTED_ANNOTATION_RELEASE="GCF_000001405.40-RS_2025_08"
MIN_LENGTH_NT=20; MAX_MODEL_PER_CLASS=3000; TEST_SIZE=.20; VAL_SIZE=.20
MMSEQS_IDENTITY=.80; MMSEQS_COVERAGE=.80
DATA_DIR=Path("ncbi_refseq_cache"); DATA_DIR.mkdir(exist_ok=True)
random.seed(SEED); np.random.seed(SEED)
print(datetime.now(timezone.utc).isoformat(), sys.version.split()[0])
bin_dir=Path.home()/".local"/"bin"; bin_dir.mkdir(parents=True,exist_ok=True)
if shutil.which("datasets") is None:
    datasets_bin=bin_dir/"datasets"
    subprocess.run(["curl","--fail","--location","--silent","--show-error",
                    "https://ftp.ncbi.nlm.nih.gov/pub/datasets/command-line/v2/linux-amd64/datasets",
                    "--output",str(datasets_bin)],check=True)
    datasets_bin.chmod(0o755)
    os.environ["PATH"]=str(bin_dir)+os.pathsep+os.environ.get("PATH","")
if shutil.which("mmseqs") is None:
    subprocess.run(["apt-get","update","-qq"],check=True)
    subprocess.run(["apt-get","install","-y","-qq","mmseqs2"],check=True)
assert "datasets" in subprocess.check_output(["datasets","--version"],text=True).lower()
print(subprocess.check_output(["datasets","--version"],text=True).strip())
print(subprocess.check_output(["mmseqs","version"],text=True).strip())


## 1. Source download and parsing

The source is NCBI Datasets human GRCh38.p14 (`GCF_000001405.40`) with `rna,gff3`; the package, CLI version, download time, and source URL are recorded in the manifest. NCBI data-use terms should be reviewed before redistribution.

In [ ]:
zip_path=DATA_DIR/"human_rna_gff3.zip"; extract=DATA_DIR/"extracted"
if not zip_path.exists():
    subprocess.run(["datasets","download","genome","accession",ACCESSION,"--include","rna,gff3","--filename",str(zip_path),"--no-progressbar"],check=True)
with zipfile.ZipFile(zip_path) as z:
    bad=z.testzip()
    assert bad is None, f"Corrupt archive member: {bad}"
    if not (extract/"ncbi_dataset"/"data"/"assembly_data_report.jsonl").exists():
        extract.mkdir(exist_ok=True)
        z.extractall(extract)
report_path=next(extract.rglob("assembly_data_report.jsonl"))
reports=[json.loads(line) for line in report_path.read_text().splitlines() if line.strip()]
assert len(reports)==1, f"Expected one assembly report, got {len(reports)}"
assembly_report=reports[0]
actual_accession=assembly_report.get("accession")
actual_assembly=assembly_report.get("assemblyInfo",{}).get("assemblyName")
actual_annotation=assembly_report.get("annotationInfo",{}).get("name")
assert actual_accession==ACCESSION, (actual_accession,ACCESSION)
assert actual_assembly==EXPECTED_ASSEMBLY, (actual_assembly,EXPECTED_ASSEMBLY)
assert actual_annotation==EXPECTED_ANNOTATION_RELEASE, (actual_annotation,EXPECTED_ANNOTATION_RELEASE)
files=list(extract.rglob("*"))
fasta=next(p for p in files if p.name.endswith((".fna",".fna.gz")))
gff=next(p for p in files if p.name.endswith((".gff",".gff3",".gff.gz",".gff3.gz")))
def ot(p): return gzip.open(p,"rt") if str(p).endswith(".gz") else open(p)
def attrs(s):
    d={}
    for item in s.strip().split(";"):
        if "=" in item:
            k,v=item.split("=",1); d[unquote(k)]=unquote(v)
    return d
ann={}; gene_biotype_by_id={}
transcript_accession_re=re.compile(r"\bN[MR]_\d+(?:\.\d+)?\b")
with ot(gff) as fh:
    for line in fh:
        if line.startswith("#"): continue
        f=line.rstrip("\n").split("\t")
        if len(f)!=9: continue
        aa=attrs(f[8]); feature=f[2].lower()
        gm=re.search(r"(?:GeneID:|gene_id=)(\d+)",f[8])
        biotype=aa.get("gene_biotype") or aa.get("gene_type") or aa.get("biotype")
        if aa.get("gbkey","").lower()=="gene" or feature in {"gene","pseudogene"}:
            if gm and biotype: gene_biotype_by_id[gm.group(1)]=biotype
            continue
        # NCBI GFF3 direct transcript rows use ID=rna-...; child rows (UTR,
        # exon, CDS, etc.) have other IDs and cannot overwrite transcript metadata.
        if not aa.get("ID","").startswith("rna-") or "transcript_id" not in aa: continue
        ids=set(transcript_accession_re.findall(aa.get("transcript_id","")+" "+aa.get("ID","")))
        if not ids: continue
        candidate={"gene_id":gm.group(1) if gm else None,
                   "gene_biotype":biotype,"gff_feature":f[2],
                   "gff_transcript_key":aa.get("gbkey")}
        for acc in ids:
            key=acc.split(".")[0]; prior=ann.setdefault(key,{})
            for k,v in candidate.items():
                if v is not None: prior[k]=v
for candidate in ann.values():
    if not candidate.get("gene_biotype") and candidate.get("gene_id"):
        candidate["gene_biotype"]=gene_biotype_by_id.get(candidate["gene_id"])
def parse_header(h):
    m=transcript_accession_re.search(h)
    if not m:return None
    accession_version=m.group(0); acc=accession_version.split(".")[0]
    p=1 if acc.startswith("NM_") else 0
    gm=re.search(r"(?:GeneID:|gene_id[= ]?)(\d+)",h,re.I)
    gs=re.search(r"\[gene=([^\]]+)\]",h)
    hb=re.search(r"\[(?:gene_biotype|gene_type|biotype)=([^\]]+)\]",h,re.I)
    return {"accession":acc,"accession_version":accession_version,"prefix_label":p,
            "gene_id_header":gm.group(1) if gm else None,
            "gene_symbol":gs.group(1) if gs else None,
            "header_biotype":hb.group(1) if hb else None,"header":h}
rows=[]; h=None; parts=[]
def flush():
    if h is None:return
    m=parse_header(h)
    if not m:return
    s="".join(parts).upper().replace("U","T")
    m.update(ann.get(m["accession"],{})); m["sequence"]=s; m["length"]=len(s)
    # Group only by NCBI GeneID; unresolved IDs are audited and excluded from modeling.
    m["gene_id"]=m.get("gene_id") or m.get("gene_id_header")
    rows.append(m)
with ot(fasta) as fh:
    for line in fh:
        line=line.strip()
        if line.startswith(">"):
            flush(); h=line; parts=[]
        else: parts.append(line)
    flush()
df_raw=pd.DataFrame(rows)
duplicate_accession_rows=int(df_raw.accession.duplicated().sum())
duplicate_accession_version_rows=int(df_raw.accession_version.duplicated().sum())
assert duplicate_accession_version_rows==0, f"Duplicate RefSeq accession.version rows: {duplicate_accession_version_rows}"
df=df_raw.reset_index(drop=True)
df["label_name"]=df.prefix_label.map({1:"protein-coding",0:"non-coding"})
df["valid_bases"]=df.sequence.str.fullmatch(r"[ACGTN]+",na=False)
print("FASTA",fasta,"GFF3",gff,"records",len(df)); print(df.label_name.value_counts())
print("Verified source:",actual_accession,actual_assembly,actual_annotation)


## 2. Data audit: transcript keys and parent-gene biotypes

The audit is calculated on every parsed record before sampling. Accession prefixes are checked against direct transcript-level GFF keys. Parent-gene biotype disagreements are reported separately because gene-level labels can differ from transcript-level RefSeq prefixes; they are review evidence, not automatic relabeling.


In [ ]:
audit={"NM_count":int((df.prefix_label==1).sum()),"NR_count":int((df.prefix_label==0).sum()),"duplicate_accession_rows":duplicate_accession_rows,"duplicate_accession_version_rows":duplicate_accession_version_rows,"missing_gff_biotype":int(df.gene_biotype.isna().sum()),"below_minimum_length":int((df.length<MIN_LENGTH_NT).sum()),"invalid_base_rows":int((~df.valid_bases).sum()),"missing_gene_id":int(df.gene_id.isna().sum())}
print(pd.Series(audit).to_string()); print("\nClass balance"); display(df.label_name.value_counts().rename_axis("label").to_frame("n").assign(fraction=lambda x:x.n/x.n.sum()))
print("\nNR_ gene_biotype values"); nr_biotypes=df.loc[df.prefix_label==0,"gene_biotype"].fillna("<missing GFF3 biotype>").value_counts().rename_axis("gene_biotype").to_frame("n"); nr_biotypes["fraction_within_NR"]=nr_biotypes.n/nr_biotypes.n.sum(); display(nr_biotypes)
def norm(x):
    if pd.isna(x):return None
    x=str(x).lower().replace("-","_").replace(" ","_")
    if x in {"protein_coding","protein_coding_gene"}:return "protein_coding"
    for key,name in [("lncrna","lncRNA"),("long_noncoding","lncRNA"),("mirna","miRNA"),("snrna","snRNA"),("snorna","snoRNA"),("trna","tRNA"),("rrna","rRNA")]:
        if key in x:return name
    return x
df["parent_gene_biotype_normalized"]=df.gene_biotype.map(norm)
df["transcript_key_normalized"]=df.gff_transcript_key.fillna("").str.lower()
# Compare transcript accession labels with transcript-level GFF keys separately
# from parent-gene biotypes; the latter can legitimately differ by transcript.
df["prefix_transcript_key_disagreement"]=((df.prefix_label==1)&(df.transcript_key_normalized!="mrna"))|((df.prefix_label==0)&(df.transcript_key_normalized=="mrna"))
df["prefix_parent_gene_biotype_disagreement"]=((df.prefix_label==1)&df.parent_gene_biotype_normalized.notna()&(df.parent_gene_biotype_normalized!="protein_coding"))|((df.prefix_label==0)&(df.parent_gene_biotype_normalized=="protein_coding"))
print("Prefix/transcript-GFF-key disagreements:",int(df.prefix_transcript_key_disagreement.sum()))
print("Prefix/parent-gene-biotype disagreements (review separately; gene-level metadata):",int(df.prefix_parent_gene_biotype_disagreement.sum()))
display(df.loc[df.prefix_transcript_key_disagreement|df.prefix_parent_gene_biotype_disagreement,["accession","prefix_label","gene_id","gff_transcript_key","gene_biotype","gff_feature","header"]].head(25))
model_pool=df[(df.length>=MIN_LENGTH_NT)&df.valid_bases&df.gene_id.notna()].copy(); print("Model pool after QC",len(model_pool))
display(model_pool.groupby("label_name").length.quantile([0,.01,.25,.5,.75,.99,1]).unstack(0))

## 3. All-transcript homology clustering and connected-component split

After balanced sampling, every sampled transcript is clustered with MMseqs2 using the explicit nucleotide database selector (`--dbtype 2`), at 80% identity and 80% coverage. The maximum sequence length is passed explicitly so long transcripts are not silently excluded. We union rows that share either an NCBI GeneID or an MMseqs2 sequence cluster; connected components are indivisible groups. Train, validation, and test are split by component, with assertions for zero component, GeneID, sequence-cluster, and exact-sequence-hash overlap. This split reduces close-sequence leakage at the chosen clustering threshold; it does not prove independence from every possible homology definition.


In [ ]:
rng=np.random.default_rng(SEED); pieces=[]
for y in [0,1]:
    p=model_pool[model_pool.prefix_label==y]
    if MAX_MODEL_PER_CLASS and len(p)>MAX_MODEL_PER_CLASS:p=p.iloc[rng.choice(len(p),MAX_MODEL_PER_CLASS,replace=False)]
    pieces.append(p)
model_df=pd.concat(pieces).sample(frac=1,random_state=SEED).reset_index(drop=True)
model_df["sequence_id"]=[f"tx_{i}" for i in range(len(model_df))]
model_df["sequence_sha256"]=model_df.sequence.map(lambda x: hashlib.sha256(x.encode("ascii")).hexdigest())
mm=DATA_DIR/"mmseqs"; mm.mkdir(exist_ok=True); transcript_fasta=DATA_DIR/"sampled_transcripts.fna"
with open(transcript_fasta,"w") as out:
    for r in model_df.itertuples():out.write(f">{r.sequence_id}\n{r.sequence}\n")
max_seq_len=int(model_df.length.max())+1
pref=mm/"transcript_clusters_80"; tsv=Path(str(pref)+"_cluster.tsv")
if not tsv.exists():
    subprocess.run(["mmseqs","easy-cluster",str(transcript_fasta),str(pref),str(mm/"transcript_tmp_80"),"--dbtype","2","--max-seq-len",str(max_seq_len),"--min-seq-id",str(MMSEQS_IDENTITY),"-c",str(MMSEQS_COVERAGE),"--cov-mode","0","--threads","2"],check=True)
cluster_map={}
with open(tsv) as f:
    for line in f:
        rep,member=line.rstrip().split("\t")[:2]; cluster_map[member]=rep
model_df["cluster_id"]=model_df.sequence_id.map(cluster_map); assert model_df.cluster_id.notna().all()

def connected_component_ids(data, sequence_cluster_col):
    parent=list(range(len(data)))
    def find(x):
        while parent[x]!=x:
            parent[x]=parent[parent[x]]; x=parent[x]
        return x
    def union(a,b):
        ra,rb=find(a),find(b)
        if ra!=rb:parent[rb]=ra
    for col in ("gene_id",sequence_cluster_col):
        first={}
        for i,value in enumerate(data[col].astype(str)):
            if value in first:union(i,first[value])
            else:first[value]=i
    roots=[find(i) for i in range(len(data))]
    return pd.Series([f"component_{root}" for root in roots],index=data.index,dtype="object")

model_df["component_id"]=connected_component_ids(model_df,"cluster_id")
from sklearn.model_selection import GroupShuffleSplit
def split3(data,seed=SEED,group_col="component_id"):
    for off in range(100):
        rs=seed+off; outer=GroupShuffleSplit(1,test_size=TEST_SIZE,random_state=rs); a,b=next(outer.split(data,groups=data[group_col])); trv=data.iloc[a]; te=data.iloc[b]
        inner=GroupShuffleSplit(1,test_size=VAL_SIZE,random_state=rs); c,d=next(inner.split(trv,groups=trv[group_col])); tr=trv.iloc[c]; va=trv.iloc[d]
        if all(set(x.prefix_label)=={0,1} for x in [tr,va,te]):return tr.copy(),va.copy(),te.copy(),rs
    raise RuntimeError("No component split with both classes")

train,val,test,split_seed=split3(model_df); sets={"train":train,"validation":val,"test":test}
for n,p in sets.items():print(n,len(p),"transcripts",p.gene_id.nunique(),"genes",p.cluster_id.nunique(),"sequence clusters",p.component_id.nunique(),"components",p.label_name.value_counts().to_dict())
pairs=[("train","validation"),("train","test"),("validation","test")]
for a,b in pairs:
    for col in ("gene_id","cluster_id","component_id","sequence_sha256"):
        overlap=len(set(sets[a][col])&set(sets[b][col])); print(a,b,col,"overlap",overlap); assert overlap==0,f"{col} overlap across {a}/{b}"
print("PASS: no shared GeneID, sequence cluster, connected component, or exact sequence across splits")


## 4. Models and validation-only tuning

The majority baseline is fit on training labels. The interpretable logistic regression uses length, GC, and normalized 3-mer frequencies. A character 3–5-mer TF-IDF logistic regression is tuned over a small `C` grid using validation AUPRC only; preprocessing is fit on training data.


In [ ]:
from sklearn.dummy import DummyClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import precision_score,recall_score,f1_score,roc_auc_score,average_precision_score,confusion_matrix
kmers=[a+b+c for a in "ACGT" for b in "ACGT" for c in "ACGT"]
def feat(ss):
    out=[]
    for s in ss:
        d=max(1,len(s)-2); c=Counter(s[i:i+3] for i in range(len(s)-2)); out.append([len(s),(s.count("G")+s.count("C"))/max(1,len(s))]+[c[k]/d for k in kmers])
    return np.asarray(out,dtype="float32")
Xtr,Xv,Xte=[feat(p.sequence) for p in [train,val,test]]
majority=DummyClassifier(strategy="most_frequent").fit(np.zeros((len(train),1)),train.prefix_label)
interp=Pipeline([("scale",StandardScaler()),("logreg",LogisticRegression(max_iter=1000,class_weight="balanced",random_state=SEED))]).fit(Xtr,train.prefix_label)
candidates={}
for c in [.25,1.,4.]:
    m=Pipeline([("tfidf",TfidfVectorizer(analyzer="char",ngram_range=(3,5),min_df=2,max_features=50000,sublinear_tf=True)),("logreg",LogisticRegression(max_iter=1000,C=c,class_weight="balanced",random_state=SEED))]).fit(train.sequence,train.prefix_label)
    candidates[c]=(average_precision_score(val.prefix_label,m.predict_proba(val.sequence)[:,1]),m)
tuned_c,(best_val_auprc,tuned)=max(candidates.items(),key=lambda x:x[1][0]); print({c:round(v[0],4) for c,v in candidates.items()},"selected C",tuned_c)
def row(name,y,p,s):return {"model":name,"precision":precision_score(y,p,zero_division=0),"recall":recall_score(y,p,zero_division=0),"f1":f1_score(y,p,zero_division=0),"auroc":roc_auc_score(y,s),"auprc":average_precision_score(y,s)}
vr=[]
for name,m,x in [("majority",majority,None),("length+GC+3mer logistic",interp,Xv),("TF-IDF logistic tuned",tuned,val.sequence)]:
    p=m.predict(np.zeros((len(val),1))) if x is None else m.predict(x); s=np.repeat(train.prefix_label.mean(),len(val)) if x is None else m.predict_proba(x)[:,1]; vr.append(row(name,val.prefix_label,p,s))
display(pd.DataFrame(vr).round(4))
trainval=pd.concat([train,val]); final=Pipeline([("tfidf",TfidfVectorizer(analyzer="char",ngram_range=(3,5),min_df=2,max_features=50000,sublinear_tf=True)),("logreg",LogisticRegression(max_iter=1000,C=float(tuned_c),class_weight="balanced",random_state=SEED))]).fit(trainval.sequence,trainval.prefix_label)

## 5. Locked test metrics, component-bootstrap intervals, and split-seed sensitivity

The final test set is touched only after model selection. The headline table includes precision, recall, F1, AUROC, AUPRC, and 95% bootstrap intervals that resample connected GeneID/sequence-similarity components, not individual transcript rows. The confusion matrix is printed. Split-seed sensitivity re-splits only the primary train+validation development records; it never uses primary test records for training or validation and evaluates validation only with the primary validation-selected `C`.


In [ ]:
def ci(y,s,groups,n=500):
    # Resample connected GeneID/sequence-similarity components to respect the
    # grouped dependence structure used by the held-out split.
    rng=np.random.default_rng(SEED); z=[]; groups=np.asarray(groups); unique_groups=np.unique(groups)
    group_rows={g:np.flatnonzero(groups==g) for g in unique_groups}
    for _ in range(n):
        sampled=rng.choice(unique_groups,size=len(unique_groups),replace=True)
        i=np.concatenate([group_rows[g] for g in sampled]); yy=np.asarray(y)[i]; ss=np.asarray(s)[i]
        if len(np.unique(yy))<2:continue
        pp=(ss>=.5).astype(int); z.append([precision_score(yy,pp,zero_division=0),recall_score(yy,pp,zero_division=0),f1_score(yy,pp,zero_division=0),roc_auc_score(yy,ss),average_precision_score(yy,ss)])
    if len(z)<20: return ["unavailable: fewer than 20 two-class component bootstrap replicates"]*5
    q=np.quantile(z,[.025,.975],axis=0); return [f"[{q[0,i]:.3f}, {q[1,i]:.3f}]" for i in range(5)]
test_rows=[]; predictions={}
for name,m,x in [("majority",majority,None),("length+GC+3mer logistic",interp,Xte),("TF-IDF logistic locked",final,test.sequence)]:
    p=m.predict(np.zeros((len(test),1))) if x is None else m.predict(x); s=np.repeat(trainval.prefix_label.mean(),len(test)) if x is None else m.predict_proba(x)[:,1]; predictions[name]=(p,s); r=row(name,test.prefix_label,p,s); q=ci(test.prefix_label,s,test.component_id)
    for k,v in zip(["precision","recall","f1","auroc","auprc"],q):r[k+"_ci95"]=v
    test_rows.append(r)
test_results=pd.DataFrame(test_rows); display(test_results.round(4)); pred,score=predictions["TF-IDF logistic locked"]; print("Confusion matrix [rows=true, cols=predicted]\n",confusion_matrix(test.prefix_label,pred))

# Diagnostic split-seed sensitivity is restricted to the primary development set.
development=pd.concat([train,val],ignore_index=True); seed_rows=[]; seen_validation_components=set()
for requested_seed in [42,43,44,45,46]:
    found=False
    for offset in range(100):
        actual_seed=requested_seed+offset
        inner=GroupShuffleSplit(1,test_size=VAL_SIZE,random_state=actual_seed)
        a,b=next(inner.split(development,groups=development.component_id)); train_s=development.iloc[a]; val_s=development.iloc[b]
        signature=frozenset(val_s.component_id.astype(str).unique())
        if signature in seen_validation_components or not all(set(x.prefix_label)=={0,1} for x in [train_s,val_s]):continue
        seen_validation_components.add(signature); found=True; break
    if not found:raise RuntimeError(f"No unique development-only validation split for seed {requested_seed}")
    m=Pipeline([("tfidf",TfidfVectorizer(analyzer="char",ngram_range=(3,5),min_df=2,max_features=50000,sublinear_tf=True)),("logreg",LogisticRegression(max_iter=1000,C=float(tuned_c),class_weight="balanced",random_state=requested_seed))]).fit(train_s.sequence,train_s.prefix_label); s=m.predict_proba(val_s.sequence)[:,1]
    seed_rows.append({"requested_split_seed":requested_seed,"actual_split_seed":actual_seed,"validation_n":len(val_s),"validation_components":val_s.component_id.nunique(),"validation_prevalence":float(val_s.prefix_label.mean()),"validation_auroc":roc_auc_score(val_s.prefix_label,s),"validation_auprc":average_precision_score(val_s.prefix_label,s)})
assert len({r["actual_split_seed"] for r in seed_rows})==len(seed_rows)
assert len({frozenset(r["actual_split_seed"] for r in seed_rows)})==1
assert len(seen_validation_components)==len(seed_rows)
display(pd.DataFrame(seed_rows).round(4))


## 6. Error analysis and limitations

The following tables break errors down by transcript length, GC content, and parent-gene biotype. A transcript-level `NR_` label can map to a gene with protein-coding biotype; that is a level-of-annotation disagreement, not proof that either annotation is wrong. The model predicts accession-prefix labels within one RefSeq snapshot, not translation or biological function. A bioinformatics reviewer should validate label semantics, release, cluster threshold, errors, and external generalization.


In [ ]:
err=test[["accession","gene_id","gene_biotype","label_name","length","sequence","header"]].copy(); err["gc_fraction"]=err.sequence.map(lambda s:(s.count("G")+s.count("C"))/max(1,len(s))); err["predicted_label"]=np.where(pred==1,"protein-coding","non-coding"); err["coding_probability"]=score; err["correct"]=err.label_name==err.predicted_label; err["error_type"]=np.select([(~err.correct)&(err.label_name=="non-coding"),(~err.correct)&(err.label_name=="protein-coding")],["false positive","false negative"],default="correct")
print("Overall error rate",round(1-err.correct.mean(),4)); display(err.groupby("label_name").agg(n=("correct","size"),error_rate=("correct",lambda x:1-x.mean()),median_length=("length","median"),median_gc=("gc_fraction","median")).round(4)); print("By GFF3 biotype (n>=5)"); display(err.groupby(err.gene_biotype.fillna("<missing>")).agg(n=("correct","size"),error_rate=("correct",lambda x:1-x.mean()),median_length=("length","median"),median_gc=("gc_fraction","median")).query("n>=5").sort_values("error_rate",ascending=False).round(4)); err["length_bin"]=pd.qcut(err.length,4,duplicates="drop"); print("By length quartile"); display(err.groupby("length_bin",observed=True).agg(n=("correct","size"),error_rate=("correct",lambda x:1-x.mean()),median_gc=("gc_fraction","median")).round(4)); err["uncertainty"]=(err.coding_probability-.5).abs(); print("Most uncertain errors"); display(err[~err.correct].sort_values("uncertainty")[["accession","gene_id","gene_biotype","label_name","predicted_label","coding_probability","length","gc_fraction","header"]].head(25))
manifest={"accession":ACCESSION,"expected_assembly":EXPECTED_ASSEMBLY,"expected_annotation_release":EXPECTED_ANNOTATION_RELEASE,"source_url":"https://www.ncbi.nlm.nih.gov/datasets/genome/"+ACCESSION+"/","download_timestamp_utc":datetime.now(timezone.utc).isoformat(),"datasets_version":subprocess.check_output(["datasets","--version"],text=True).strip(),"mmseqs_version":subprocess.check_output(["mmseqs","version"],text=True).strip(),"min_length_nt":MIN_LENGTH_NT,"max_model_per_class":MAX_MODEL_PER_CLASS,"mmseqs_min_seq_id":MMSEQS_IDENTITY,"mmseqs_min_coverage":MMSEQS_COVERAGE,"audit":audit,"nr_biotypes":nr_biotypes.reset_index().to_dict("records"),"prefix_transcript_key_disagreements":int(df.prefix_transcript_key_disagreement.sum()),"prefix_parent_gene_biotype_disagreements":int(df.prefix_parent_gene_biotype_disagreement.sum()),"split_seed":int(split_seed),"train_n":len(train),"validation_n":len(val),"test_n":len(test),"cross_split_gene_overlap":0,"cross_split_cluster_overlap":0,"modeling_sample_per_class_cap":MAX_MODEL_PER_CLASS,"modeling_sample_label_counts":model_df.label_name.value_counts().to_dict(),"test_label_prevalence":float(test.prefix_label.mean()),"source_zip_sha256":hashlib.sha256(zip_path.read_bytes()).hexdigest()}
Path("transcript_coding_status_manifest.json").write_text(json.dumps(manifest,indent=2,default=str)); print(json.dumps(manifest,indent=2,default=str))

## 7. Similarity-threshold limitation

The primary holdout groups every sampled transcript at 80% nucleotide identity and 80% coverage, then joins those clusters with shared GeneIDs. This notebook does not run a multi-threshold sweep: that would repeat full transcript clustering several times and is outside this bounded study. Results should therefore be interpreted relative to this explicit grouping rule, and a future robustness study can compare alternative thresholds and an external RefSeq release.


In [ ]:
# Save aggregate audit/results only; do not export transcript sequences or row-level predictions.
manifest["mmseqs_dbtype"]=2
manifest["mmseqs_identity"]=MMSEQS_IDENTITY
manifest["mmseqs_coverage"]=MMSEQS_COVERAGE
manifest["mmseqs_max_seq_len"]=max_seq_len
manifest["primary_split_seed"]=split_seed
manifest["primary_split_counts"]={name:{"transcripts":int(len(part)),"genes":int(part.gene_id.nunique()),"sequence_clusters":int(part.cluster_id.nunique()),"components":int(part.component_id.nunique()),"labels":{str(k):int(v) for k,v in part.prefix_label.value_counts().items()}} for name,part in sets.items()}
manifest["primary_model_test_results"]=test_results.to_dict("records")
manifest["test_confusion_matrix_tfidf"]=confusion_matrix(test.prefix_label,pred).tolist()
manifest["test_prevalence"]=float(test.prefix_label.mean())
manifest["exact_sequence_hash_overlap"]={f"{a}_vs_{b}":len(set(sets[a].sequence_sha256)&set(sets[b].sequence_sha256)) for a,b in pairs}
manifest["connected_component_overlap"]={f"{a}_vs_{b}":len(set(sets[a].component_id)&set(sets[b].component_id)) for a,b in pairs}
manifest["validation_seed_sensitivity"]=seed_rows
manifest["scope_note"]="One annotation snapshot and one MMseqs2 identity/coverage threshold; no FLIP2, variants, functional validation, or external-cohort test."
manifest_path=Path("transcript_coding_status_manifest.json"); manifest_path.write_text(json.dumps(manifest,indent=2,default=str))
result_dir=Path("transcript_coding_status_results"); result_dir.mkdir(exist_ok=True)
test_results.to_csv(result_dir/"primary_test_metrics.csv",index=False)
nr_biotypes.reset_index().to_csv(result_dir/"noncoding_biotype_counts.csv",index=False)
with zipfile.ZipFile("transcript_coding_status_results.zip","w",zipfile.ZIP_DEFLATED) as z:
    z.write(manifest_path,manifest_path.name)
    z.write(result_dir/"primary_test_metrics.csv","results/primary_test_metrics.csv")
    z.write(result_dir/"noncoding_biotype_counts.csv","results/noncoding_biotype_counts.csv")
print("RUN_COMPLETE",actual_annotation,"test prevalence",manifest["test_prevalence"])
print("RESULT_BUNDLE",Path("transcript_coding_status_results.zip").stat().st_size)
